<a href="https://colab.research.google.com/github/A-ros1076/BUS118s/blob/Dev/Group_Exercise_Agentic_AI_in_Customer_Service_and_Sales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# ============================================================
# Dyson Customer Service Chatbot
# Course: BUS118s - AI and Business Applications
# API: Google Gemini


import os
import re
# Reverted import back to google.generativeai as genai for GenerativeModel compatibility
import google.generativeai as genai

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    # Not running in Colab — read from system environment variable instead
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

# Stop immediately if no key is found — nothing works without it
if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY not found.\n"
        "In Colab: click the key icon (Secrets) and add GEMINI_API_KEY.\n"
        "Locally: run  export GEMINI_API_KEY='your-key'  in your terminal."
    )

# Register the API key with the Gemini library
genai.api_key = GEMINI_API_KEY


# ── STEP 2: System Prompt ───────────────────────────────
# The system prompt is sent to the model before the conversation starts.
# It defines WHO the chatbot is, WHAT it knows, and HOW it should behave.
# This directly satisfies Rubric #2 (Clear Chatbot Role).

SYSTEM_PROMPT = """
You are Alex, a friendly and knowledgeable customer service assistant for Dyson.
Dyson is a premium home technology brand known for vacuums, air purifiers,
fans, and hair care products (Airwrap, Supersonic hairdryer).

You are trained to handle three types of customer inquiries:

1. ORDER STATUS
   - Ask the customer for their order number (format: DYS-XXXXXX).
   - Standard shipping: 2-3 business days to process, 5-7 business days to arrive.
   - Express shipping: 1-2 business days to arrive.
   - Provide reassurance and next steps if an order seems delayed.

2. REFUND & RETURN POLICY
   - Dyson offers a 30-day return window from the date of purchase.
   - Items must be returned in original condition with all original packaging.
   - Refunds are processed within 5-10 business days after Dyson receives the return.
   - Gift purchases extend the return window to 60 days.
   - Damaged or defective products may qualify for an immediate replacement.

3. PRODUCT RECOMMENDATIONS
   Help customers choose the right Dyson product based on their needs.
   Key products and prices:
   - V15 Detect Absolute: Best overall cordless vacuum, laser dust detection -- $749
   - V12 Detect Slim: Lightweight cordless vacuum, great for smaller homes -- $649
   - Cyclone V10: Reliable cordless vacuum, strong value -- $499
   - Purifier Cool Formaldehyde: Air purifier + cooling fan, removes gases -- $649
   - Airwrap Multi-Styler: Styles, curls, and dries hair with no extreme heat -- $599
   - Supersonic Hairdryer: Fast, gentle drying that reduces heat damage -- $429

Always be polite, concise, and professional.
If the customer's question is unclear, ask one clarifying question before answering.
Do NOT make up order details -- if you cannot look something up, say so honestly
and guide the customer to contact Dyson support at 1-866-693-9766.
"""


# ── STEP 3: Intent Detection (Agent Logic) ─────────────────────────────
# This is the "agent" part of the chatbot.
# Before sending each message to Gemini, we classify what the user is
# asking about. This routing logic satisfies Rubric #5 (Basic Agent Logic).

def detect_intent(user_input):
    """
    Classify the user's message into one of three inquiry types.

    We scan the message for topic-specific keywords and return
    a label that tells us which service category applies.

    Parameters:
        user_input (str): The raw message typed by the user.

    Returns:
        str: One of 'order_status', 'refund_policy',
             'product_recommendation', or 'general'.
    """
    text = user_input.lower()  # Lowercase so keyword matching is case-insensitive

    # Keywords that suggest the customer is asking about their order
    order_keywords = [
        "order", "track", "tracking", "shipped", "shipping",
        "delivery", "deliver", "package", "arrive", "arrival", "dys-"
    ]

    # Keywords that suggest the customer wants to return something or get a refund
    refund_keywords = [
        "refund", "return", "money back", "cancel", "exchange",
        "policy", "send back", "damaged", "broken", "warranty", "defective"
    ]

    # Keywords that suggest the customer wants help choosing a product.
    # Expanded to cover air quality, pollutants, and setup-related queries
    # so those conversations route correctly instead of falling to 'general'.
    product_keywords = [
        "recommend", "suggestion", "suggest", "which", "best",
        "vacuum", "purifier", "airwrap", "hairdryer", "hair dryer",
        "fan", "buy", "purchase", "product", "model", "compare",
        "difference", "v15", "v12", "v10", "supersonic",
        "air", "quality", "pollutant", "pollutants", "allergen",
        "allergens", "formaldehyde", "filter", "filtration",
        "set up", "setup", "install", "app", "connect", "wifi"
    ]

    # Check in priority order and return the first match
    if any(word in text for word in order_keywords):
        return "order_status"
    elif any(word in text for word in refund_keywords):
        return "refund_policy"
    elif any(word in text for word in product_keywords):
        return "product_recommendation"
    else:
        return "general"  # Catch-all for greetings, thanks, unclear questions


# Map each intent to a routing tag we prepend to the user message.
# The model never sees this tag in its printed output -- it is only used
# internally to steer the model toward the right knowledge area.
INTENT_LABELS = {
    "order_status":           "[ROUTING -> Order Status]",
    "refund_policy":          "[ROUTING -> Refund/Return Policy]",
    "product_recommendation": "[ROUTING -> Product Recommendation]",
    "general":                "[ROUTING -> General Inquiry]"
}


# ── STEP 4: Initialize Gemini Model ─────────────────────────────
# We create a GenerativeModel with our system prompt baked in.
# gemini-flash-latest is fast and well-suited for conversational tasks.

# List available models to confirm API access is working
print("Available Gemini models:")
for m in genai.list_models():
    if "generateContent" in m.supported_generation_methods:
        print(m.name)

model = genai.GenerativeModel(
    model_name="models/gemini-2.5-flash",
    system_instruction=SYSTEM_PROMPT  # Role is defined here at the model level
)

# start_chat() creates a stateful chat session.
# Every time we call chat_session.send_message(), Gemini automatically
# includes ALL previous messages in the request -- this is how multi-turn
# conversation memory works. Satisfies Rubric #4 (Conversation Memory).
chat_session = model.start_chat(history=[])


# ── STEP 5: Response Cleaner ────────────────────────────────────
def clean_response(text):
    """
    Strip markdown formatting from the model's response so the output
    prints as plain readable text in the terminal or Colab console.

    Gemini sometimes returns **bold**, *italic*, or bullet * markers.
    We remove those characters so they do not appear as raw symbols.

    Parameters:
        text (str): Raw response string from the Gemini API.

    Returns:
        str: Clean plain-text version of the response.
    """
    # Strip bullet-point asterisks first (standalone * at line start, no closing pair).
    # Must run before the bold/italic rule so these lone * do not get misread
    # as one half of a paired **bold** match across lines.
    text = re.sub(r'^\s*\*+\s+', '  - ', text, flags=re.MULTILINE)
    # Remove **bold** and *italic* paired markers (keep the inner text).
    # No DOTALL -- keeping . line-scoped prevents cross-line false matches.
    text = re.sub(r'\*{1,2}(.+?)\*{1,2}', r'\1', text)
    return text.strip()



# ── STEP 6: Message Handler ─────────────────────────
def send_message(user_input):
    """
    Process one turn of the conversation:
      1. Detect the intent of the user's message.
      2. Prepend a routing tag so the model knows the topic.
      3. Send the augmented message to Gemini (history is auto-included).
      4. Return the cleaned plain-text response.

    Parameters:
        user_input (str): The user's raw message.

    Returns:
        str: Plain-text response from the assistant.
    """
    # Classify the question type
    intent = detect_intent(user_input)

    # Build the message with routing context prepended.
    # Example: "[ROUTING -> Refund/Return Policy] I want to return my vacuum"
    # The routing tag guides the model internally -- it is never printed to the user.
    tagged_message = f"{INTENT_LABELS[intent]} {user_input}"

    # Send to Gemini -- chat_session includes all prior turns automatically
    response = chat_session.send_message(tagged_message)

    # Strip markdown symbols before returning
    return clean_response(response.text)


# ── STEP 7: Conversation Loop ─────────────────────────
def main():
    """
    Run the chatbot interactively in the terminal or Colab.
    Keeps looping until the user types 'quit' or 'exit'.
    """
    print("\n" + "=" * 60)
    print("       Dyson Customer Service Assistant")
    print("       Powered by Google Gemini AI")
    print("=" * 60)
    print("  Ask about: orders | returns | product recommendations")
    print("  Type 'quit' to end the conversation.\n")

    while True:
        # Get input from the user
        user_input = input("You: ").strip()

        # Exit commands
        if user_input.lower() in ["quit", "exit", "bye", "goodbye"]:
            print("\nDyson Assistant: Thank you for contacting Dyson support. "
                  "Have a great day!")
            break

        # Skip blank input
        if not user_input:
            print("(Please type a message to continue.)\n")
            continue

        # Send the message and print the clean response
        response = send_message(user_input)
        print(f"\nDyson Assistant: {response}\n")


# Standard Python entry point -- runs main() when the script is executed directly
if __name__ == "__main__":
    main()

Available Gemini models:
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025

       Dyson Customer Servic